# Feature Engineering

Ableitung des finalen Feature-Sets für die Modellierung — basierend auf den Erkenntnissen aus der Analyse-Phase (`03_analysis_*`).

## Warum ein eigenes Notebook?

Die Analyse-Phase (`03_analysis_*`) hat Erkenntnisse geliefert, die das ursprüngliche Feature-Set
aus `02_preparation.ipynb` erweitern und präzisieren. Dieses Notebook führt alle Entscheidungen
zusammen und produziert den finalen Train/Test-Datensatz für die Modellierung.

> **Iterativer Prozess — dokumentiert:**
> `02_preparation` = Cleaning auf Basis der EDA
> `04_feature_engineering` = Feature-Auswahl auf Basis der Analyse
>
> Was sich geändert hat und warum steht im `PROCESS_LOG.md`.

## Bereinigungsstrategie — Finale Filter

Drei Datenqualitätsprobleme aus der Analyse die **vor** dem Feature Engineering angewendet werden.

### 🔴 Filter 1 — `canceled == True` entfernen

**Warum:** Die VBZ hat ihre Definition von „Ausfall" im Juli 2024 geändert.
- Vor Juli 2024: Kurzwendungen und Teilausfälle zählten als `canceled = True`
- Ab Juli 2024: Nur echte Komplett-Ausfälle sind `canceled = True`

Das Ergebnis: Die Zahlen vor und nach Juli 2024 sind nicht vergleichbar. Wer `canceled` einfach
drin lässt, misst keine Betriebsqualität — sondern eine Definitionsänderung.
→ `canceled == False` für alle Delay-Analysen und das Modell. → **F-TARGET-05**

---

### 🔴 Filter 2 — Nov/Dez 2025 GTFS-Artefakt

**Warum:** VBZ publiziert GTFS-Daten wöchentlich (donnerstags). Für den grossen Fahrplanwechsel
Dezember 2025 („Tramnetz Süd") wurden neue Fahrplan-Daten bereits ab Oktober/November 2025
schrittweise eingespeist. Das bedeutet: Die Solldaten stimmen nicht mehr mit dem tatsächlichen
Betrieb überein. Gemessene Verspätung = (Ist-Zeit) − (falscher Soll-Wert) → Verzerrung −0.8s.
→ `NOT (year == 2025 AND month >= 11)` aus Train + Test. → **F-TARGET-06**

---

### 🔴 Filter 3 — Linie E ausschliessen

**Warum:** Linie E ist eine Entlastungs-/Verstärkerlinie — nur bei Bedarf (Grossevents,
Stosszeiten) eingesetzt. Sie ist planmässig im GTFS modelliert, kann aber per Natur keine festen
Fahrzeiten einhalten. OTP: 56.2%, Ø Delay: 128–130s (Netzschnitt: ~56s).

Ein Modell das Linie E mittrainiert, lernt systematisch falsche Muster — oder wird von ihr
dominiert. Im Report wird der Ausschluss explizit begründet.
→ `line_name != "E"`. → **F-TARGET-12, F-NET-08**

---

### 🟡 Kontext-Filter — Starthaltestellen für Reporting-Metriken

**Warum:** Erste Haltestelle jeder Fahrt (`stop_sequence == 1`) hat eingebauten Fahrplan-Puffer.
Das Tram wartet dort auf seine planmässige Abfahrtszeit — negative Delay-Werte sind kein
Betriebssignal sondern Wartezeit. Für **Reporting-Metriken** (Netz-Durchschnitt, OTP) gilt:
`stop_sequence > 1`. Für das **Modell** bleiben alle Stops drin — `stop_sequence` wird als Feature
kodiert, damit das Modell den Starthalte-Effekt selbst lernen kann.

→ Für Metriken: `stop_sequence > 1`. Für Modell-Training: alle Stops + `stop_sequence` als Feature.

## Setup

In [ ]:
from zh_tram_flow.notebook import *
TRAIN, TEST, lf = setup_analysis("04_feature_engineering")

lf_all   = pl.concat([pl.scan_parquet(TRAIN), pl.scan_parquet(TEST)])

%load_ext autoreload
%autoreload 2

## Schritt 1 — Filter anwenden

Alle drei 🔴-Filter sequenziell.

In [ ]:
section_header("Filter — Canceled / GTFS-Artefakt / Linie E")

n_raw = lf_all.select(pl.len()).collect().item()
log(f"Roh: {n_raw:,} Zeilen")

lf_filtered = (
    lf_all
    # Filter 1: Nur echte Fahrten (kein Datendefinitions-Artefakt)
    .filter(pl.col("canceled") == False)
    # Filter 2: GTFS-Artefakt Nov/Dez 2025 entfernen
    .filter(
        ~(
            (pl.col("operating_date").dt.year() == 2025) &
            (pl.col("operating_date").dt.month() >= 11)
        )
    )
    # Filter 3: Linie E ausschliessen (strukturell nicht vergleichbar)
    .filter(pl.col("line_name") != "E")
)

n_filtered = lf_filtered.select(pl.len()).collect().item()
n_removed  = n_raw - n_filtered
log(f"Nach Filter: {n_filtered:,} Zeilen")
log(f"Entfernt:    {n_removed:,} Zeilen ({n_removed/n_raw:.1%})")

## Schritt 2 — Feature-Entscheidungen aus der Analyse

Übersicht aller Features — geordnet nach erwarteter Vorhersagekraft (aus den Analyse-Findings).

### Zeit-Features
| Feature | Typ | Quelle | Begründung | Finding |
|:---|:---|:---|:---|:---|
| `hour` | Int8 | `arrival_schedule` | Stärkster temporaler Effekt — 21h Peak, kein Morgenrush | F-TEMP-01 |
| `weekday` | Int8 (0=Mo) | `arrival_schedule` | Do schlechtester Tag (60.4s), So bester (48.4s) | F-TEMP-02 |
| `month` | Int8 | `arrival_schedule` | November-Peak (67–73s), starke Saisonalität | F-TEMP-05 |
| `season` | Int8 (1=Winter…4=Herbst) | `month` | Herbst schlechteste Saison, Winter beste | F-TEMP-06 |
| `is_november` | Bool | `month == 11` | Stärkster Monatseffekt — separates Flag nützlich | F-TEMP-05 |
| `is_weekend` | Bool | `weekday >= 5` | Sa kaum besser als Werktag; So deutlich besser | F-TEMP-04 |
| `is_rush_hour` | Bool | `hour ∈ {17,18,19}` | Abend-Rush real (+11s); Morgen-Rush nicht! | F-TEMP-01 |
| `gtfs_year` | Str (`j23`/`j24_j25`) | `operating_date` | Netzwechsel Dez 2023 — schwacher aber realer Effekt (+0.5s netzweit) | F-NET-03 |

### Räumliche Features
| Feature | Typ | Quelle | Begründung | Finding |
|:---|:---|:---|:---|:---|
| `line_name` | Cat | direkt | Stärkster räumlicher Prädiktor — L11 68.7s vs. Netz-Ø | F-SPAT-05 |
| `stop_name` | Cat (Target-Enc.) | direkt | Periphere Hotspots (Enzenbühl 93.8s) — n-Threshold nötig | F-SPAT-01 |
| `district_nr` | Int8 | direkt | Kreis 11 worst (68.3s), Kreis 5 best (49.9s) — additiv nützlich | F-SPAT-03 |
| `stop_sequence` | Int16 | direkt | Starthalte-Puffer kodieren; Endhalte-Signal | Target-Notebook |

### Wetter-Features
| Feature | Typ | Quelle | Begründung | Finding |
|:---|:---|:---|:---|:---|
| `has_snow` | Bool | direkt | Stärkster Wettereffekt: +54.0s, OTP −10.9pp | F-WEAT-01 |
| `precipitation` | Float | direkt | Dosis-Wirkungs: <2mm=62.6s → >10mm=89.5s; r=0.036 | F-WEAT-02 |
| `has_rain` | Bool | direkt | +8.9s, OTP −3.1pp | F-WEAT-02 |
| `has_heavy_rain` | Bool | direkt | +23.3s, OTP −7.6pp | F-WEAT-02 |
| `temperature` | Float | direkt | Schwacher Effekt (r=0.018), nicht-linear | F-WEAT-04 |
| `is_hot` | Bool | `temperature > 20` | +2.0s Delta — schwach aber real | F-WEAT-04 |
| ~~`is_windy`~~ | ~~Bool~~ | ~~direkt~~ | **ENTFERNT** — NaN überall, nie befüllt | F-WEAT-03 |

### Event-Features
| Feature | Typ | Quelle | Begründung | Finding |
|:---|:---|:---|:---|:---|
| `is_holiday` | Bool | direkt | Stärkstes negatives Signal: −9.9s (Feiertag = bester Tag) | F-EVNT-01 |
| `event_weight` | Int8 (0–3) | direkt | Gross=+10.5s, Mittel=+2.7s, Klein≈Normal (+0.05s) | F-EVNT-02 |
| `has_event` | Bool | `event_weight > 0` | Basisindikator | F-EVNT-02 |
| `event_type` | Cat | direkt | Fachmessen worst (66.0s); Super League near Normal | F-EVNT-04 |

### Interaktions-Features (neu aus Analyse)
| Feature | Formel | Begründung | Finding |
|:---|:---|:---|:---|
| `event_weight × hour` | Produkt / Interaktion | Event-Effekt primär abends (18–22h) — Interaktion aussagekräftiger als Haupteffekt | F-EVNT-03 |
| `is_school_holiday` | Extern (ZH Schulkalender) | Schulferien-Täler im Rolling-Average deutlich sichtbar | F-TEMP-08 |

### Features die entfernt werden
| Feature | Grund |
|:---|:---|
| `is_windy` | NaN überall — nie korrekt befüllt |
| `is_rush_hour` (7–9h) | Morgenrush existiert nicht im Delay-Signal — 7h unter Netz-Ø |
| `dwell_time` (kontinuierlich) | 71.3% = 0s, kein Zusammenhang mit Delay messbar |


## Schritt 3 — Feature Engineering implementieren

> **TODO:** Code aus `02_preparation.ipynb` Feature-Engineering-Sektion hierher übertragen und mit den Analyse-Erkenntnissen erweitern.

In [ ]:
section_header("Feature Engineering")

# TODO: Features aus 02_preparation übernehmen + erweitern

# Zeit-Features (aus 02_preparation bereits vorhanden)
# lf_feat = lf_filtered.with_columns([
#     pl.col("arrival_schedule").dt.hour().cast(pl.Int8).alias("hour"),
#     pl.col("arrival_schedule").dt.weekday().cast(pl.Int8).alias("weekday"),
#     pl.col("arrival_schedule").dt.month().cast(pl.Int8).alias("month"),
#     ...
# ])

# NEU aus Analyse:
# .with_columns([
#     # gtfs_year
#     pl.when(pl.col("operating_date") < pl.lit("2024-01-01").str.to_date())
#       .then(pl.lit("j23")).otherwise(pl.lit("j24_j25")).alias("gtfs_year"),
#
#     # is_november (stärkster Monats-Flag)
#     (pl.col("operating_date").dt.month() == 11).alias("is_november"),
#
#     # is_hot (>20°C — schwacher aber realer Effekt)
#     (pl.col("temperature") > 20).alias("is_hot"),
#
#     # Rush-Hour korrigiert (nur Abend — Morgen hat keinen Delay-Peak)
#     pl.col("arrival_schedule").dt.hour().is_in([17,18,19]).alias("is_rush_hour"),
#
#     # Interaktion event × abend
#     (pl.col("event_weight") * pl.col("arrival_schedule").dt.hour().is_in(range(18,23)).cast(pl.Int8))
#       .alias("event_hour_interaction"),
# ])

log("TODO: Feature Engineering Code implementieren")

## Schritt 4 — Export für Modellierung

In [ ]:
section_header("Export — Train / Test Feature Set")

# TODO: Finalen Feature-Set exportieren
# out_train = "data/processed/train_final.parquet"
# out_test  = "data/processed/test_final.parquet"
# lf_feat_train.sink_parquet(out_train)
# lf_feat_test.sink_parquet(out_test)
# log(f"Exportiert: {out_train}")
# log(f"Exportiert: {out_test}")

log("TODO: Export implementieren — danach 05_modeling.ipynb starten")

## Übergang zur Modellierung

Mit dem exportierten Feature-Set ist `05_modeling.ipynb` bereit zum Starten.

**Modell-Kandidaten (aus Analyse begründet):**
- **LightGBM / XGBoost** — wegen Interaktionseffekten (hour × event, line × season)
- **Baseline:** OTP-Rate als Random-Guess-Benchmark — 87% muss übertroffen werden

**Ziel-Metrik:**
- Primär: **MAE** (Mean Absolute Error) — robuster gegenüber Extremwerten
- Sekundär: **OTP-Accuracy** (Anteil korrekt als on-time/delayed klassifiziert)

**Bekannte Herausforderungen:**
- Klassenungleichgewicht bei Events (Gross-Events n=724k vs. Normal 70.5M) → F-EVNT-05
- Extremwerte bis +5000s → Robust-Modell oder Log-Transform — F-TARGET-07
- Linie E ausgeschlossen → im Report dokumentieren